In [1]:
def read_dataframe(filename):
    columns = [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance"
    ]

    df = pd.read_parquet(filename, columns=columns)
    df=df.head(1000)  # For testing purposes, limit to first 1000 rows
    df["duration"] = (
        df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    ).dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ["PULocationID", "DOLocationID"]
    numerical = ["trip_distance"]

    df[categorical] = df[categorical].astype(str)

    return df

In [2]:
import numpy as np
import pickle

In [3]:
import pandas as pd
import sklearn
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Lasso
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("new")

<Experiment: artifact_location='/workspaces/MLOPs/01-intro/mlruns/1', creation_time=1789148677294, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789148677294, lifecycle_stage='active', name='new', tags={}, trace_location=None, workspace='default'>

In [4]:
df_train=read_dataframe("./yellow_tripdata_2026-01.parquet")

In [5]:
categorical=["PULocationID","DOLocationID"]
numerical=["trip_distance"]
dv=DictVectorizer()
train_dict=df_train[categorical+numerical].to_dict(orient="records")
X_train=dv.fit_transform(train_dict)


In [6]:
y_train=df_train["duration"]

In [7]:
X_train.indices = X_train.indices.astype(np.int32)
X_train.indptr = X_train.indptr.astype(np.int32)

In [8]:
with mlflow.start_run():
    mlflow.set_tag("developper","hakim")
    alpha=.002
    mlflow.log_param("alpha",alpha)
    lr=Lasso(alpha=alpha)
    lr.fit(X_train,y_train)
    y_pred=lr.predict(X_train)
    mae=mean_absolute_error(y_train,y_pred)
    mlflow.log_metric("mae",mae)
    mlflow.log_artifact(local_path="../models/lin_reg.bin",artifact_path="models_pickle")

In [9]:
with open("../models/lin_reg.bin","wb") as f_out:
    pickle.dump((dv,lr),f_out)

In [11]:
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [12]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model","xgboost")
        mlflow.log_params(params)

        best_params = {
            "learning_rate": 0.9919286966376871,
            "max_depth": 67,
            "min_child_weight": 0.8916854838870615,
            "objective": "reg:squarederror",
            "reg_alpha": 0.008594124238282383,
            "reg_lambda": 0.042113157151565286
        }
        mlflow.log_params(best_params)
        booster = xgb.train(
            params=params,
            dtrain=xgb.DMatrix(X_train, label=y_train),
            num_boost_round=100,
            evals=[(xgb.DMatrix(X_train, label=y_train), "train")],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(xgb.DMatrix(X_train))
        mae = mean_absolute_error(y_train, y_pred)


        mlflow.log_metric("mae", mae)
        mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

    return {"loss": mae, "status": STATUS_OK}

In [13]:
search_space = {
    "max_depth": scope.int(hp.quniform("max_depth", 4, 100, 1)),
    "learning_rate": hp.loguniform("learning_rate", -3, 0),
    "reg_alpha": hp.loguniform("reg_alpha", -5, -1),
    "reg_lambda": hp.loguniform("reg_lambda", -6, -1),
    "min_child_weight": hp.loguniform("min_child_weight", -1, 3),
    "objective": "reg:squarederror",
}
best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials(),
)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

job exception: Changing param values is not allowed. Params were already logged='[{'key': 'learning_rate', 'old_value': '0.2116703245957639', 'new_value': '0.9919286966376871'}, {'key': 'max_depth', 'old_value': '85', 'new_value': '67'}, {'key': 'min_child_weight', 'old_value': '4.490643892133809', 'new_value': '0.8916854838870615'}, {'key': 'reg_alpha', 'old_value': '0.054610990596691826', 'new_value': '0.008594124238282383'}, {'key': 'reg_lambda', 'old_value': '0.00419256382101573', 'new_value': '0.042113157151565286'}]' for run ID='e365e35d08a446579a3713612534fdb2'.



  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]


MlflowException: Changing param values is not allowed. Params were already logged='[{'key': 'learning_rate', 'old_value': '0.2116703245957639', 'new_value': '0.9919286966376871'}, {'key': 'max_depth', 'old_value': '85', 'new_value': '67'}, {'key': 'min_child_weight', 'old_value': '4.490643892133809', 'new_value': '0.8916854838870615'}, {'key': 'reg_alpha', 'old_value': '0.054610990596691826', 'new_value': '0.008594124238282383'}, {'key': 'reg_lambda', 'old_value': '0.00419256382101573', 'new_value': '0.042113157151565286'}]' for run ID='e365e35d08a446579a3713612534fdb2'.

In [17]:
with mlflow.start_run():
        mlflow.set_tag("model","xgboost")

        best_params = {
            "learning_rate": 0.9919286966376871,
            "max_depth": 67,
            "min_child_weight": 0.8916854838870615,
            "objective": "reg:squarederror",
            "reg_alpha": 0.008594124238282383,
            "reg_lambda": 0.042113157151565286
        }
        mlflow.log_params(best_params)
        booster = xgb.train(
            params=best_params,
            dtrain=xgb.DMatrix(X_train, label=y_train),
            num_boost_round=100,
            evals=[(xgb.DMatrix(X_train, label=y_train), "train")],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(xgb.DMatrix(X_train))
        mae = mean_absolute_error(y_train, y_pred)

        with open("../models/xgb_model.bin", "wb") as f_out:
            pickle.dump(dv, f_out)
        mlflow.log_metric("mae", mae)
        mlflow.log_artifact(local_path="../models/xgb_model.bin", artifact_path="models_pickle")
        mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")    

[0]	train-rmse:0.82795
[1]	train-rmse:0.17074
[2]	train-rmse:0.15878
[3]	train-rmse:0.15852
[4]	train-rmse:0.15847
[5]	train-rmse:0.15846
[6]	train-rmse:0.15845
[7]	train-rmse:0.15844
[8]	train-rmse:0.15844
[9]	train-rmse:0.15844
[10]	train-rmse:0.15844
[11]	train-rmse:0.15844
[12]	train-rmse:0.15844
[13]	train-rmse:0.15844
[14]	train-rmse:0.15844
[15]	train-rmse:0.15844
[16]	train-rmse:0.15844
[17]	train-rmse:0.15844
[18]	train-rmse:0.15844
[19]	train-rmse:0.15844
[20]	train-rmse:0.15844
[21]	train-rmse:0.15844
[22]	train-rmse:0.15844
[23]	train-rmse:0.15844
[24]	train-rmse:0.15844
[25]	train-rmse:0.15844
[26]	train-rmse:0.15844
[27]	train-rmse:0.15844
[28]	train-rmse:0.15844
[29]	train-rmse:0.15844
[30]	train-rmse:0.15844
[31]	train-rmse:0.15844
[32]	train-rmse:0.15844
[33]	train-rmse:0.15844
[34]	train-rmse:0.15844
[35]	train-rmse:0.15844
[36]	train-rmse:0.15844
[37]	train-rmse:0.15844
[38]	train-rmse:0.15844
[39]	train-rmse:0.15844
[40]	train-rmse:0.15844
[41]	train-rmse:0.15844
[4

2026/09/12 10:12:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [16]:
mlflow.autolog(disable=True)

In [18]:
logged_model="runs:/ae2c0d339ba94611843849ed4e45a1f6/models_mlflow"
loaded_model = mlflow.pyfunc.load_model(logged_model)

In [19]:
loaded_model

mlflow.pyfunc.loaded_model:
  artifact_path: /workspaces/MLOPs/01-intro/mlruns/1/models/m-fd380c13205d4a6db6e17d956bde7363/artifacts
  flavor: mlflow.xgboost
  run_id: ae2c0d339ba94611843849ed4e45a1f6

In [20]:
xgboost_model=mlflow.xgboost.load_model(logged_model)


In [21]:
xgboost_model

In [22]:
xgboost_model.predict(xgb.DMatrix(X_train))

array([ 5.5549235,  5.7171426,  8.881725 , 42.799553 , 13.500369 ,
       13.599484 , 10.630138 , 24.616564 , 37.7329   ,  9.582433 ,
       36.199486 , 12.483995 , 27.682947 ,  1.9506497,  4.617155 ,
       21.933325 , 26.26826  , 41.953976 ,  9.966627 ,  6.5646887,
        4.6831927,  4.2160606, 22.551907 ,  8.04829  ,  7.916733 ,
       38.182915 ,  5.182784 ,  6.7282143, 15.598589 , 15.18268  ,
       18.73763  ,  9.382558 , 12.065534 , 17.833652 ,  7.2997913,
       25.583742 , 19.546734 , 11.549755 , 46.95081  , 30.433306 ,
        7.90009  , 10.883348 , 50.729885 ,  4.9009395, 23.032583 ,
       17.580948 ,  6.187775 ,  5.2827206,  6.8288093, 16.54735  ,
       43.048958 , 53.81189  , 25.282436 , 21.084345 , 16.283693 ,
       22.366589 , 10.902911 , 21.333271 , 38.115543 , 14.030471 ,
        8.600625 , 11.717032 ,  4.1377935, 25.848907 , 26.151518 ,
       23.749956 , 30.449903 , 24.349392 , 30.398878 , 24.016499 ,
       39.949894 , 27.500584 ,  8.298139 ,  2.5165095,  2.5165